# Import Libraries

In [46]:
import numpy as np
import pandas as pd

import string                                 # for punctuation removal, and lowercase conversion

# For NLP
import nltk
nltk.download('punkt_tab')                    # for tokenization
nltk.download('stopwords')                    # for stopwords
from nltk.stem import PorterStemmer           # for stemming
nltk.download('wordnet')                      # for lemmataization
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import CountVectorizer     # for BoW = since

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

from sklearn.feature_extraction.text import TfidfVectorizer

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


# Reading Dataset

In [47]:
filepath = "/content/drive/MyDrive/ICT_AIML Class/Datasets/spam.xlsx"
df_spam = pd.read_excel(filepath)
df_spam.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


# EDA

In [48]:
df_spam.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   v1          5572 non-null   object
 1   v2          5572 non-null   object
 2   Unnamed: 2  50 non-null     object
 3   Unnamed: 3  12 non-null     object
 4   Unnamed: 4  6 non-null      object
dtypes: object(5)
memory usage: 217.8+ KB


In [49]:
# Inference:
  # No missing values in the relevant 2 columns

# relevant data is present only in first 2 columns only
df_spam = df_spam[['v1', 'v2']]

# after dropping
df_spam.shape

(5572, 2)

In [50]:
df_spam.shape

(5572, 2)

In [51]:
# class imbalance in target column
df_spam['v1'].value_counts()

,count
v1,
ham,4825
spam,747


In [83]:
# Defining target column
target = 'v1'

# Preprocessing

In [52]:
# Removing Special Characters

def remove_punctuation(text):
  punctuationless_text = ''.join([i for i in text if i not in string.punctuation])
  # comparing each char in text against the punctuations list
  return punctuationless_text

In [53]:
# Convert to Lowercase
  # Using function .lower() from Class String

In [54]:
# convert the entire text into a list of unique token words
# Function to get tokens from the sentence

def tokenization(text):
  words_list = nltk.word_tokenize(text)
  return words_list

In [55]:
# function to remove stopwords from any text
def remove_stopwords(list_of_tokens):
  stop_words= nltk.corpus.stopwords.words('english')
  tokens_without_stopwords = [word for word in list_of_tokens if word not in stop_words]
  return tokens_without_stopwords

In [56]:
# stem object
stem_obj = PorterStemmer()      # shows all steming options for a word

# function for stemming
def stemming(list_of_cleartokens):
  stem_list = [stem_obj.stem(word) for word in list_of_cleartokens]
  return stem_list

In [57]:
# lemmatization object
lemma_obj = WordNetLemmatizer()

# function to apply lemmatization
def lemmatization(list_of_stemmedwords):
  list_of_lemmawords = [lemma_obj.lemmatize(word) for word in list_of_stemmedwords]
  return list_of_lemmawords

## Applying all preprocessing steps

In [58]:
df_spam['v2'] = df_spam['v2'].astype(str)

# Punctuation Removal
punctuationless_texts = []
for text in df_spam['v2']:
    punctuationless_texts.append(remove_punctuation(text))

df_spam['punctuationless_text'] = punctuationless_texts


# Lowercasing
lowercase_texts = []
for text in df_spam['punctuationless_text']:
    lowercase_texts.append(text.lower())

df_spam['lowercase_text'] = lowercase_texts


# Tokenization
tokenized_texts = []
for text in df_spam['lowercase_text']:
    tokenized_texts.append(tokenization(text))

df_spam['tokens'] = tokenized_texts


# Stopword Removal
stopwords_removed_lists = []
for token_list in df_spam['tokens']:
    stopwords_removed_lists.append(remove_stopwords(token_list))

df_spam['removed_stopwords'] = stopwords_removed_lists

# Stemming
stemmed_token_lists = []
for token_list in df_spam['removed_stopwords']:
    stemmed_token_lists.append(stemming(token_list))

df_spam['stemmed_tokens'] = stemmed_token_lists


# Lemmatization
lemmatized_token_lists = []
for token_list in df_spam['stemmed_tokens']:
    lemmatized_token_lists.append(lemmatization(token_list))
df_spam['lemmatized_tokens'] = lemmatized_token_lists


# Printing final df after preprocessing
display(df_spam.head())

,v1,v2,punctuationless_text,lowercase_text,tokens,removed_stopwords,stemmed_tokens,lemmatized_tokens
0,ham,"Go until jurong point, crazy.. Available only ...",Go until jurong point crazy Available only in ...,go until jurong point crazy available only in ...,"[go, until, jurong, point, crazy, available, o...","[go, jurong, point, crazy, available, bugis, n...","[go, jurong, point, crazi, avail, bugi, n, gre...","[go, jurong, point, crazi, avail, bugi, n, gre..."
1,ham,Ok lar... Joking wif u oni...,Ok lar Joking wif u oni,ok lar joking wif u oni,"[ok, lar, joking, wif, u, oni]","[ok, lar, joking, wif, u, oni]","[ok, lar, joke, wif, u, oni]","[ok, lar, joke, wif, u, oni]"
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in 2 a wkly comp to win fa cup fina...,"[free, entry, in, 2, a, wkly, comp, to, win, f...","[free, entry, 2, wkly, comp, win, fa, cup, fin...","[free, entri, 2, wkli, comp, win, fa, cup, fin...","[free, entri, 2, wkli, comp, win, fa, cup, fin..."
3,ham,U dun say so early hor... U c already then say...,U dun say so early hor U c already then say,u dun say so early hor u c already then say,"[u, dun, say, so, early, hor, u, c, already, t...","[u, dun, say, early, hor, u, c, already, say]","[u, dun, say, earli, hor, u, c, alreadi, say]","[u, dun, say, earli, hor, u, c, alreadi, say]"
4,ham,"Nah I don't think he goes to usf, he lives aro...",Nah I dont think he goes to usf he lives aroun...,nah i dont think he goes to usf he lives aroun...,"[nah, i, dont, think, he, goes, to, usf, he, l...","[nah, dont, think, goes, usf, lives, around, t...","[nah, dont, think, goe, usf, live, around, tho...","[nah, dont, think, goe, usf, live, around, tho..."


# Final Preprocessed Text  
To be taken as the feature for training and texting

In [59]:
# Create a new column containing all the data after final preprocessing (i.e., lemmatization) as a text items intead of lists

df_spam['clean_text'] = df_spam['lemmatized_tokens'].apply(lambda x:" ".join(x))
df_spam['clean_text'].head()

,clean_text
0,go jurong point crazi avail bugi n great world...
1,ok lar joke wif u oni
2,free entri 2 wkli comp win fa cup final tkt 21...
3,u dun say earli hor u c alreadi say
4,nah dont think goe usf live around though


# Label Encoding  
Encoding the target column before splitting the dataset does not cause data leakage  as:
* One-to-One Mapping: Standard label encoding or integer mapping for a target column creates a strict, deterministic, one-to-one transformation.   
For example, converting ['Cat', 'Dog', 'Cat'] to [0, 1, 0] relies strictly on independent row values.
* No Information Bleeding: The encoded value of any row does not depend on the distribution or statistical metrics of other rows in the dataset.

In [85]:
# Label encoding the entire target column
le_obj = LabelEncoder()

df_spam[target] = le_obj.fit_transform(df_spam[target])

# After label encoding
df_spam[target].head()

,v1
0,0
1,0
2,1
3,0
4,0


# Train Test Split

In [86]:
# Checking value counts of target column
df_spam[target].value_counts()

,count
v1,
0,4825
1,747


In [87]:
# Checking ratio of classes in target column
class_counts = df_spam[target].value_counts()

majority = class_counts.max()
minority = class_counts.min()

print(f"Ham : Spam = {majority}:{minority}")

ratio = majority / minority

print(f"Class imbalance ratio = {ratio:.2f}:1")

Ham : Spam = 4825:747
Class imbalance ratio = 6.46:1


The target is heavily imbalanced  => 85:15 = not-spam:spam  

Can cause unusal high accuracy, as model identifies more no of not-spam. Can be overcome by:  
1. **Deleting existing data** from class having higher ratio imbalance - Leads to data loss in small datasets

2. **Augmenting/Creating new data** for class having lower imbalance - Leads to duplicate rows which isn't beneficial for the model learning

3. Using **Boosting Algorithms** - Better option

In [88]:
# Train Test Split

X = df_spam['clean_text']
y = df_spam[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state = 42,
    stratify=y                  # for ensuring distribution of target classes due to class imbalance
)

In [90]:
# Verifying
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

(4457,)
(4457,)
(1115,)
(1115,)


# Bag of Words (Count Vectorization)

In [91]:
# obj
count_vec_obj = CountVectorizer()

# applying using fit_transform()
X_train_bow = count_vec_obj.fit_transform(X_train)
X_test_bow = count_vec_obj.transform(X_test)

#### For understanding what happens in BoW

Count Vectorization (BoW) returns a **sparse matrix of word counts**  
* Each columns -  is every unique word present over the entire **Corpus** (Collection of SMS)  
* Each **value in the matrix** - the corresponding count (frequency) of each unique word in each **Document** (each SMS/text message)
* Each row - represents a vector, containing n column as dimensions that define the vector ?

In [92]:
X_train_bow   # cannot be printed as it is a very large sparse matrix

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 37462 stored elements and shape (4457, 7076)>

In [93]:
# So, convert sparse matrix to array for better understanding
X_train_bow.toarray()

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [1, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [94]:
print(X_train_bow.shape)
print(X_test_bow.shape)

# No of rows -> 457, 1115 is the no of text msgs in X_train and X_test
# no of cols => 7971 is the 'no of bags' => no of unique words over all these 5572 text msgs = nunique words over the entire corpus  =>

# so we have 5572 numerical vectors in this corpus built over these 7971 count values

(4457, 7076)
(1115, 7076)


# Classification (Binary)

## Boosting

## Logistic Regression

In [70]:
# creating object
log_reg = LogisticRegression()

# train the model
log_reg.fit(X_train, y_train)

# make predictions on X_test
y_pred_log_reg = log_reg.predict(X_test)

### Evaluation

In [71]:
# evaluate the model by comparing with y_test
print("Evaluating Logistic Regression Model:")

print("\nAccuracy :", accuracy_score(y_test, y_pred_log_reg))
print("Precision :", precision_score(y_test, y_pred_log_reg))
print("Recall :", recall_score(y_test, y_pred_log_reg))
print("F1 Score :", f1_score(y_test, y_pred_log_reg))

clsf_log_reg = classification_report(y_test,y_pred_log_reg)
print("\nClassification Report :")
print(clsf_log_reg)

conf_log_reg = confusion_matrix(y_test, y_pred_log_reg)
print("\nConfusion Matrix :")
print(conf_log_reg)

Evaluating Logistic Regression Model:

Accuracy : 0.9820627802690582
Precision : 1.0
Recall : 0.8657718120805369
F1 Score : 0.9280575539568345

Classification Report :
              precision    recall  f1-score   support

           0       0.98      1.00      0.99       966
           1       1.00      0.87      0.93       149

    accuracy                           0.98      1115
   macro avg       0.99      0.93      0.96      1115
weighted avg       0.98      0.98      0.98      1115


Confusion Matrix :
[[966   0]
 [ 20 129]]


In [63]:
# So the array of vectors or of numerical features can be fed into an ML model to classify into target => 'Spam' or 'Ham'

# TrainTestSplit for TF-IDF

In [73]:
X_tfidf = df_spam['clean_text']
y = df_spam['v1']

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [80]:
# Verifying the train test split
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

(4457,)
(4457,)
(1115,)
(1115,)


# TF-IDF

In [81]:
# before tf-idf
X_train.head()

,clean_text
184,go noth greatby
2171,wont wat wit guy
5422,ok ksri knw 2 sivatat askd
4113,stand away doesnt heart ach without dont wonde...
4588,finish work yet someth


In [74]:
# Create TF-IDF object
tfidf = TfidfVectorizer(max_features=7000)

# Convert text into TF-IDF features

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [75]:
# After TF-IDF transformation we get a sparse matrix
X_train_tfidf

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 37386 stored elements and shape (4457, 7000)>

E.g.,
```
| free | entry | meeting | prize | win  |
| ---- | ----- | ------- | ----- | -----|
| 0.68 | 0.52  | 0       | 0     | 0.41 |
| 0    | 0     | 0.74    | 0     | 0    |
| 0.32 | 0     | 0       | 0.81  | 0    |

```

* Every row is one SMS.
* Every column is one vocabulary word.
* Every value is the TF-IDF weight.

This matrix is what machine learning models use.

In [ ]:
# Now the text has been converted into numerical features, so you can train a classifier.